# Chapter 37: Feature Detection, Description & Matching
## Multimedia & Computer Vision Learning Hub - ITERA
**Author:** Aji Pamungkas Tri Nurcahyo  
**Institution:** Institut Teknologi Sumatera (ITERA)  
**Objective:** Mempelajari deteksi titik sudut Harris, ekstraksi deskriptor SIFT/ORB, Lowe's Ratio Test, dan penolakan outlier RANSAC.

### 1. Landasan Teori & Formulasi Matematis
Fungsi respon sudut Harris dirumuskan sebagai:
$$R = \det(\mathbf{M}) - k \cdot [\text{trace}(\mathbf{M})]^2$$
Di mana $\mathbf{M} = \sum w(x,y) \begin{bmatrix} I_x^2 & I_x I_y \\ I_x I_y & I_y^2 \end{bmatrix}$ adalah matriks momen kedua autokorelasi lokal.

Jumlah iterasi RANSAC untuk rasio outlier $\epsilon$ dan probabilitas sukses $p$:
$$N = \frac{\ln(1 - p)}{\ln(1 - (1 - \epsilon)^s)}$$

In [ ]:
# 2. Dependency Setup & Environment
import numpy as np
import cv2
print(f"NumPy Version: {np.__version__}")
print(f"OpenCV Version: {cv2.__version__}")

In [ ]:
# 3. Sintesis Pasangan Citra dengan Transformasi Geometri Diketahui
def create_transformed_pair():
    img1 = np.zeros((300, 300), dtype=np.uint8)
    # Buat beberapa bentuk geometris geometris unik
    cv2.rectangle(img1, (50, 50), (120, 150), 200, -1)
    cv2.circle(img1, (200, 100), 40, 150, -1)
    cv2.putText(img1, "ITERA", (80, 240), cv2.FONT_HERSHEY_SIMPLEX, 1.2, 255, 2)
    
    # Transformasikan img1 dengan rotasi 25 derajat dan translasi
    M_rot = cv2.getRotationMatrix2D((150, 150), 25, 0.9)
    img2 = cv2.warpAffine(img1, M_rot, (300, 300))
    return img1, img2, M_rot

img1, img2, M_gt = create_transformed_pair()
print(f"Ukuran Citra 1: {img1.shape}, Citra 2: {img2.shape}")

In [ ]:
# 4. Ekstraksi Fitur & Pencocokan ORB + Brute-Force Hamming Distance
orb = cv2.ORB_create(nfeatures=200)
kp1, des1 = orb.detectAndCompute(img1, None)
kp2, des2 = orb.detectAndCompute(img2, None)

print(f"Keypoints terdeteksi Citra 1: {len(kp1)}, Citra 2: {len(kp2)}")

# Pencocokan KNN (k=2) untuk rasio test
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
matches_knn = bf.knnMatch(des1, des2, k=2)

# Terapkan Lowe's Ratio Test (tau = 0.75)
good_matches = []
for m_pair in matches_knn:
    if len(m_pair) == 2:
        m, n = m_pair
        if m.distance < 0.75 * n.distance:
            good_matches.append(m)

print(f"Total Pasangan Mentah: {len(matches_knn)}, Pasangan Lolos Ratio Test: {len(good_matches)}")

In [ ]:
# 5. Estimasi Matriks Transformasi Menggunakan RANSAC
if len(good_matches) >= 4:
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    
    H_est, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 3.0)
    inliers = mask.ravel().tolist().count(1)
    print(f"Jumlah Inliers RANSAC: {inliers} dari {len(good_matches)} pasang ({inliers/len(good_matches)*100:.1f}%)")
    print("Matriks Homografi Terestimasi:\n", H_est)
else:
    print("Jumlah kecocokan tidak mencukupi untuk estimasi homografi")

### 6. Batasan & Analisis Kegagalan (Failure Modes)
1. **Pencahayaan Berubah Signifikan**: Jarak Hamming ORB peka terhadap pergeseran biner jika iluminasi berubah non-monotonik.
2. **Tekstur Lemah atau Homogen**: Daerah datar menghasilkan determinan matriks autokorelasi mendekati 0 sehingga tidak ada titik sudut yang terdeteksi.
3. **Skala Perubahan Terlalu Ekstrem**: Jika perbesaran melebihi kapasitas piramida skala ORB, deskriptor gagal menemukan tetangga terdekat.

### 7. Referensi Akademik
- Lowe, D. G. (2004). *Distinctive Image Features from Scale-Invariant Keypoints*. IJCV.
- Rublee, E., et al. (2011). *ORB: An efficient alternative to SIFT or SURF*. ICCV.
- Fischler, M. A., & Bolles, R. C. (1981). *Random Sample Consensus*. CACM.